# N/LAB ML Practical: Neural Networks

## Create functions to build a temporal datasets for transactional data

## Important: You must use a Serverless Base enviroment v4 and add tensorflow as one of the depdendancies

Today we will be using a dataset that "contains household level transactions over two years from a group of 2,500 households who are frequent shoppers at a retailer.". This is the same data as the last practical.


The table of interest for today is:<br>
`transaction_data`

See the instructions below for connecting to the database, listing tables etc.

### The database documentation
The [full documentation](https://drive.google.com/file/d/1Ioozg-V5NJB5wDwoPXDhdZEbClABRZqd)

The ER diagram:

![databasedocumentation](https://www.nlab.org.uk/wp-content/uploads/completejourney.png)

#### Recall the interpretation of a one-to-many relationship.

**Rows in the "many" table are extended with the information for the one row that is matched in the "one" table.**

NOTE: In this data the basket and basket_items tables are combined and we just have a transaction_data table. Make sure you understand what a line in this table represents before you write any queries!

## Task:
Build a dataset to predict spend (over a specified period, i.e. weekly) per household.

In [0]:
import urllib
exec(urllib.request.urlopen('https://drive.google.com/uc?export=download&id=1RBJW7od_upD0O-671cH9_RB2n5n_wLSx').read())
if not spark.catalog.tableExists("transaction_data_small"):
  spark.sql("CREATE TABLE transaction_data_small AS SELECT t.* FROM transaction_data t JOIN (SELECT * FROM hh_demographic ORDER BY household_key LIMIT 500) USING (household_key)")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 32.8 MB/s eta 0:00:00
Initaializing completejourney database.... on first run this will take ~5min.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Done.


This notebook contains two parts and covers two weeks worth of practicals.<br>
<b>Part 1:</b> The first practical.<br>

<b>Part 2:</b> The second practical.

### The task

This practical we will consider predicting monthly (4 week) per household spend (low/med/high) using neural networks.

### Data
We will use **a reduced data set** from Practical 4 and 7, specifically only the first 500 households. This is in the table:
`transaction_data_small`

It is located in an sqlite database which you can download with the code and link provided for you in a cell in this notebook.

We will use a modified version (provided) of functions created in practical 4 to generate our training and test sets. While this has been provided for you in this practical it is expected that you understand how it works. This is the same function as was provided in practical 7.

### Data documentation
The full documentation can be found [here](https://drive.google.com/file/d/1Ioozg-V5NJB5wDwoPXDhdZEbClABRZqd/view). Note the full data source [is avaliable onine](https://www.dunnhumby.com/careers/engineering/sourcefiles). However, please use the copy already loaded into your database.

### Documentation for Keras and TensorFlow v2.
See the [Slides](https://docs.google.com/presentation/d/146heD7t8mFxX8Dem9D3UFum_Q790c_fRSJT86fcNxE4/edit?usp=sharing).

---
#**Part 0: Code provided for you**
Read and understand

---

# Helper function

In [0]:
def get_dataset( reference_day = 697, tumbling_window_size = 28, output_window_size = 28 ):
    sql = """
    SELECT household_key,
           CAST(:ref_date AS INT) AS ref_day,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) AND day <= CAST(:ref_date AS INT) + CAST(:ows AS INT) THEN sales_value ELSE 0 END) as output_feature,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT) AND day <= CAST(:ref_date AS INT) THEN sales_value ELSE 0 END ) as f1,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*2 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT) THEN sales_value ELSE 0 END ) as f2,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*3 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*2 THEN sales_value ELSE 0 END ) as f3,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*4 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*3 THEN sales_value ELSE 0 END ) as f4,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*5 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*4 THEN sales_value ELSE 0 END ) as f5,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*6 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*5 THEN sales_value ELSE 0 END ) as f6,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*7 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*6 THEN sales_value ELSE 0 END ) as f7,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*8 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*7 THEN sales_value ELSE 0 END ) as f8,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*9 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*8 THEN sales_value ELSE 0 END ) as f9,
           SUM(CASE WHEN day > CAST(:ref_date AS INT) -CAST(:ws AS INT)*10 AND day <= CAST(:ref_date AS INT)-CAST(:ws AS INT)*9 THEN sales_value ELSE 0 END ) as f10
    FROM transaction_data_small
    WHERE day <= 700
    GROUP BY household_key
    """

    df = spark.sql(sql, {'ref_date': reference_day, 'ws': tumbling_window_size, 'ows': output_window_size}).toPandas()

    return df.drop(columns = ['ref_day','output_feature', 'household_key'], inplace = False), df.output_feature

# Data loading and preparation

## Task 1: Load a single hold-out test set, including a validation set.

Recall there are 711 days in the data. However, we need all periods to be fully covered (there is very little data in the last 28 day period since only 11 days are within the data range). Therefore our effective data size is 700 days.

The function, given a reference date in days, returns 10 tumbling window features, where each feature is an aggregate (SUM) over 28 days.
The output feature is the aggregate over the 28 days immediatly proceeding the reference date. It only uses the first 700 days. 

If you still unsure how to do this ask and/or read this [story about Bob and Jane](https://docs.google.com/document/d/1wIZARovA8NXOJ2OBjAW7Lw5QjtovDHlLbdxl6VAMW0M/edit?usp=sharing).

### Task 1a: Given the above information, select the reference date for the test set and create the variables X_test y_test

### Task 1b: Given the above information, select the reference date for the validation set and create the variables X_valid y_valid

### Task 1c: Select the the reference date for the training set and create the variables X_train, y_train


In [0]:
X_test, y_test = get_dataset( reference_day = 700-(1*28), tumbling_window_size = 28, output_window_size = 28 )

In [0]:
X_valid, y_valid = get_dataset( reference_day = 700-(2*28), tumbling_window_size = 28, output_window_size = 28 )

In [0]:
X_train, y_train = get_dataset( reference_day = 700-(3*28), tumbling_window_size = 28, output_window_size = 28 )

## Task 2: Convert the output feature in the train, validation and test sets to be low/mid/high
Use the following rules:
low <= 30<br>
30 < mid <= 100<br>
high > 100


In [0]:
y_train[y_train <= 30] = 0
y_train[ (y_train > 30) & (y_train <=100) ] = 1
y_train[ y_train > 100] = 2

y_test[y_test <= 30] = 0
y_test[ (y_test > 30) & (y_test <= 100) ] = 1
y_test[y_test > 100] = 2

y_valid[y_valid <= 30] = 0
y_valid[ (y_valid > 30) & (y_valid <= 100) ] = 1
y_valid[y_valid > 100] = 2

---
#**Part 1 (Your turn)**
---

## Task 1: Train a deep neural net (fully connected layers). Your choices:
* Number of layers
* Number of nodes per layer
* Batch size
* Number of iterations

In all cases (for now) please:
* use an relu activation function for all but the last layer
* use a softmax activation function for the last layer
* provide the labels as integers (rather than one hot encoding) 
* compile your model via:
```
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["sparse_categorical_accuracy"])
```

**What's the best accuracy you can get on the validation set (at least 60% should be possible)?**

Remember you can make a summary of your model and see a graphical representation via:
```
model.summary()
keras.utils.plot_model(model, "my_model.png", show_shapes=True)
```


In [0]:
# You will need to import tensorflow, so this has been done for you
# If you get an error regarding tensorflow now being found, 
# make sure you read the IMPORTANT note at the top of this file!
import tensorflow as tf

---
#**Part 2: (second practical session)**
---

## Task 2: Refine your model

Copy your model below.

Now try and improve it by:
* changing the initialization of the weights in the layers
* changing the activation function
* including batch normalization

**What helped? What didn't? Is it a combination? Does the improvement worth it (considering the score on the validation set)?**

**Note:** If you want to see what weights have been learnt you can call:
```
model.weights
```
after training.

## Task 2b: Test the two models (from Tasks 3 & 4) on the test set
What was the difference in performance? **NOTE: this is in general bad practice if we were going to then select and use one of these models.** Why? As we are procedurally overfitting (selecting based on knowledge from the test set). However, for now it serves to show to you while you learn that the model is actually doing better on completely unseen instances.

## Task 3: Try adding regularitsation to your model to improve it
As per the slides, I'd recommend trying a version of dropout. 

**HINT:** you may need to include more neurons per layer. 

Why? Your problem may have high complexity. Removing the model's ability to model this (via dropout) may hurt.

## Task 3b: Test all the models so far on the test set
**Once again note: this is in general bad practice if we were going to then select and use one of these models.** Why? As we are procedurally overfitting (selecting based on knowledge from the test set). However, for now it serves to show to you while you learn that the model is actually doing better on completely unseen instances.

## Task 4: Add early stopping
**HINT 1:** Think about what you are monitoring (if you want the validation version prepend the name with 'val_')

**HINT 2:** Set the min_delta by considering the performance of past models.

**HINT 3:** In the specific version of tensorflow we're using (2.0) the import is:
`from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping`
This has changed in tensorflow 2.1

### Considering just the validation scores, did that help?

## Task 4b: Test all the models so far on the test set
**Once again note: this is in general bad practice if we were going to then select and use one of these models.** Why? As we are procedurally overfitting (selecting based on knowledge from the test set). However, for now it serves to show to you while you learn that the model is actually doing better on completely unseen instances.

## Task 5: Change the optimizer
Slide example:
```
# Momentum optimizer
optimizer = keras.optimizers.SGD(lr=0.001, momentum=0.9)

# AdaGrad
optimizer = keras.optimizers.Adagrad(lr=0.001)

# ADAM

optimizer = keras.optimizers.Adam(lr=0.001, beta_1=0.9, beta_2=0.999)

# Using the optimizer
model.compile(
loss="sparse_categorical_crossentropy",
     optimizer=optimizer,
     metrics=["accuracy"])

```

## Considering just the validation scores, what did you learn?

## Task 5b: Test all the models so far on the test set
**Once again note: this is in general bad practice if we were going to then select and use one of these models.** Why? As we are procedurally overfitting (selecting based on knowledge from the test set). However, for now it serves to show to you while you learn that the model is actually doing better on completely unseen instances.

In [0]:
w